In [1]:
import math
import time
import gc
from pathlib import Path
from typing import Dict, List, Optional

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn import BatchNorm1d
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import roc_auc_score, f1_score, mean_squared_error, accuracy_score
from sklearn.preprocessing import LabelEncoder, StandardScaler

import shap

# Проверка GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [2]:
import os
import random
import numpy as np

def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    try:
        import torch
        torch.manual_seed(seed)
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)  # для multi-GPU
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
    except ImportError:
        pass
    try:
        import tensorflow as tf
        tf.random.set_seed(seed)
    except ImportError:
        pass

seed_everything(42)

## KAN

In [3]:
class KANLinear(nn.Module):
    def __init__(
        self,
        in_features,
        out_features,
        grid_size=5,
        spline_order=3,
        scale_noise=0.1,
        scale_base=1.0,
        scale_spline=1.0,
        enable_standalone_scale_spline=True,
        base_activation=torch.nn.SiLU,
        grid_eps=0.02,
        grid_range=[-1, 1],
    ):
        super().__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.grid_size = grid_size
        self.spline_order = spline_order

        h = (grid_range[1] - grid_range[0]) / grid_size
        grid = (
            (torch.arange(-spline_order, grid_size + spline_order + 1) * h + grid_range[0])
            .expand(in_features, -1)
            .contiguous()
        )
        self.register_buffer("grid", grid)

        self.base_weight = nn.Parameter(torch.Tensor(out_features, in_features))
        self.spline_weight = nn.Parameter(
            torch.Tensor(out_features, in_features, grid_size + spline_order)
        )
        if enable_standalone_scale_spline:
            self.spline_scaler = nn.Parameter(torch.Tensor(out_features, in_features))

        self.scale_noise = scale_noise
        self.scale_base = scale_base
        self.scale_spline = scale_spline
        self.enable_standalone_scale_spline = enable_standalone_scale_spline
        self.base_activation = base_activation()
        self.grid_eps = grid_eps
        self.reset_parameters()

    def reset_parameters(self):
        nn.init.kaiming_uniform_(self.base_weight, a=math.sqrt(5) * self.scale_base)
        with torch.no_grad():
            noise = ((torch.rand(self.grid_size + 1, self.in_features, self.out_features) - 0.5) *
                     self.scale_noise / self.grid_size)
            self.spline_weight.data.copy_(
                (self.scale_spline if not self.enable_standalone_scale_spline else 1.0) *
                self.curve2coeff(self.grid.T[self.spline_order:-self.spline_order], noise)
            )
            if self.enable_standalone_scale_spline:
                nn.init.kaiming_uniform_(self.spline_scaler, a=math.sqrt(5) * self.scale_spline)

    def b_splines(self, x):
        assert x.dim() == 2 and x.size(1) == self.in_features
        grid = self.grid
        x = x.unsqueeze(-1)
        bases = ((x >= grid[:, :-1]) & (x < grid[:, 1:])).to(x.dtype)
        for k in range(1, self.spline_order + 1):
            bases = ((x - grid[:, :-(k + 1)]) / (grid[:, k:-1] - grid[:, :-(k + 1)])) * bases[:, :, :-1] + \
                    ((grid[:, k + 1:] - x) / (grid[:, k + 1:] - grid[:, 1:(-k)])) * bases[:, :, 1:]
        return bases.contiguous()

    def curve2coeff(self, x, y):
        A = self.b_splines(x).transpose(0, 1)
        B = y.transpose(0, 1)
        solution = torch.linalg.lstsq(A, B).solution
        return solution.permute(2, 0, 1).contiguous()

    @property
    def scaled_spline_weight(self):
        if self.enable_standalone_scale_spline:
            return self.spline_weight * self.spline_scaler.unsqueeze(-1)
        return self.spline_weight

    def forward(self, x):
        assert x.dim() == 2 and x.size(1) == self.in_features
        base_output = F.linear(self.base_activation(x), self.base_weight)
        spline_output = F.linear(
            self.b_splines(x).view(x.size(0), -1),
            self.scaled_spline_weight.view(self.out_features, -1)
        )
        return base_output + spline_output

    @torch.no_grad()
    def update_grid(self, x, margin=0.01):
        splines = self.b_splines(x).permute(1, 0, 2)
        orig_coeff = self.scaled_spline_weight.permute(1, 2, 0)
        unreduced_spline_output = torch.bmm(splines, orig_coeff).permute(1, 0, 2)
        x_sorted = torch.sort(x, dim=0)[0]
        batch = x.shape[0]
        grid_adaptive = x_sorted[torch.linspace(0, batch - 1, self.grid_size + 1, dtype=torch.int64, device=x.device)]
        uniform_step = (x_sorted[-1] - x_sorted[0] + 2 * margin) / self.grid_size
        grid_uniform = (torch.arange(self.grid_size + 1, dtype=torch.float32, device=x.device).unsqueeze(1) *
                        uniform_step + x_sorted[0] - margin)
        grid = self.grid_eps * grid_uniform + (1 - self.grid_eps) * grid_adaptive
        grid = torch.cat([
            grid[:1] - uniform_step * torch.arange(self.spline_order, 0, -1, device=x.device).unsqueeze(1),
            grid,
            grid[-1:] + uniform_step * torch.arange(1, self.spline_order + 1, device=x.device).unsqueeze(1)
        ], dim=0)
        self.grid.copy_(grid.T)
        self.spline_weight.data.copy_(self.curve2coeff(x, unreduced_spline_output))

    def regularization_loss(self, regularize_activation=1.0, regularize_entropy=1.0):
        l1_fake = self.spline_weight.abs().mean(-1)
        reg_act = l1_fake.sum()
        p = l1_fake / reg_act
        reg_ent = -torch.sum(p * p.log())
        return regularize_activation * reg_act + regularize_entropy * reg_ent

In [4]:
class KAN(nn.Module):
    def __init__(self, layers_hidden, grid_size=5, spline_order=3,
                 scale_noise=0.1, scale_base=1.0, scale_spline=1.0,
                 base_activation=torch.nn.SiLU, grid_eps=0.02, grid_range=[-1, 1]):
        super().__init__()
        self.layers = nn.ModuleList()
        for in_f, out_f in zip(layers_hidden, layers_hidden[1:]):
            self.layers.append(KANLinear(
                in_f, out_f, grid_size=grid_size, spline_order=spline_order,
                scale_noise=scale_noise, scale_base=scale_base, scale_spline=scale_spline,
                base_activation=base_activation, grid_eps=grid_eps, grid_range=grid_range
            ))

    def forward(self, x, update_grid=False):
        for layer in self.layers:
            if update_grid:
                layer.update_grid(x)
            x = layer(x)
        return x

    def regularization_loss(self, regularize_activation=1.0, regularize_entropy=1.0):
        return sum(l.regularization_loss(regularize_activation, regularize_entropy) for l in self.layers)


class ColumnEmbedding(nn.Module):
    """Обучаемое встраивание для каждого категориального признака."""
    def __init__(self, vocabulary: Dict[str, Dict[str, int]], embedding_dim: int):
        super().__init__()
        self.columns = list(vocabulary.keys())
        self.embeddings = nn.ModuleDict()
        for col in self.columns:
            num_cats = len(vocabulary[col])
            self.embeddings[col] = nn.Embedding(num_cats, embedding_dim)

    def forward(self, x, column_name):
        # x: [batch] индексы категорий
        return self.embeddings[column_name](x)

In [5]:
class FeatureWiseKAN(nn.Module):
    """Один KAN на каждый непрерывный признак."""
    def __init__(self, num_features, hidden_dim=8, grid_size=5, spline_order=3):
        super().__init__()
        self.num_features = num_features
        self.models = nn.ModuleList([
            KAN([1, hidden_dim, 1], grid_size=grid_size, spline_order=spline_order)
            for _ in range(num_features)
        ])

    def forward(self, x):
        effects = []
        for i, model in enumerate(self.models):
            xi = x[:, i:i+1]
            effects.append(model(xi))
        return torch.cat(effects, dim=1)


class FeatureWiseMLP(nn.Module):
    """Один MLP на каждый непрерывный признак (baseline)."""
    def __init__(self, num_features, hidden_dim=16):
        super().__init__()
        self.num_features = num_features
        self.models = nn.ModuleList([
            nn.Sequential(
                nn.Linear(1, hidden_dim),
                nn.GELU(),
                nn.Linear(hidden_dim, hidden_dim),
                nn.GELU(),
                nn.Linear(hidden_dim, 1)
            )
            for _ in range(num_features)
        ])

    def forward(self, x):
        effects = []
        for i, model in enumerate(self.models):
            xi = x[:, i:i+1]
            effects.append(model(xi))
        return torch.cat(effects, dim=1)


class CategoricalMainEffects(nn.Module):
    """Обучаемая таблица эффектов для категориальных признаков."""
    def __init__(self, vocabulary: Dict[str, Dict[str, int]]):
        super().__init__()
        self.columns = list(vocabulary.keys())
        self.effects = nn.ModuleDict()
        for col in self.columns:
            n_categories = len(vocabulary[col])
            self.effects[col] = nn.Embedding(n_categories, 1)

    def forward(self, x):
        # x: [batch, num_cat] – индексы категорий
        effects = []
        for i, col in enumerate(self.columns):
            xi = x[:, i].long()
            effects.append(self.effects[col](xi))
        return torch.cat(effects, dim=1)


class ContinuousTokenEncoder(nn.Module):
    """Проецирует каждый непрерывный признак в token размерности embedding_dim."""
    def __init__(self, num_features, embedding_dim):
        super().__init__()
        self.num_features = num_features
        self.projection = nn.ModuleList([
            nn.Sequential(
                nn.Linear(1, embedding_dim),
                nn.LayerNorm(embedding_dim)
            )
            for _ in range(num_features)
        ])

    def forward(self, x):
        tokens = []
        for i, layer in enumerate(self.projection):
            xi = x[:, i:i+1]
            tokens.append(layer(xi).unsqueeze(1))
        return torch.cat(tokens, dim=1)

In [ ]:
# class InterpretableTabTransformer(nn.Module):
#     def __init__(
#         self,
#         output_dim,
#         vocabulary,
#         num_continuous_features,
#         embedding_dim=32,
#         nhead=4,
#         num_layers=2,
#         dim_feedforward=128,
#         dropout=0.1,
#         main_effect_type="kan",      # "kan" или "mlp"
#         kan_hidden_dim=8,
#         use_transformer=True         # если False, то только аддитивная часть
#     ):
#         super().__init__()
#         self.output_dim = output_dim
#         self.vocabulary = vocabulary
#         self.num_continuous_features = num_continuous_features
#         self.embedding_dim = embedding_dim
#         self.num_categorical_features = len(vocabulary)
#         self.main_effect_type = main_effect_type
#         self.use_transformer = use_transformer

#         # Нормализация непрерывных
#         self.continuous_norm = BatchNorm1d(num_continuous_features)

#         # Непрерывные main‑эффекты
#         if main_effect_type == "kan":
#             self.continuous_main_effect = FeatureWiseKAN(
#                 num_continuous_features, hidden_dim=kan_hidden_dim
#             )
#         else:
#             self.continuous_main_effect = FeatureWiseMLP(
#                 num_continuous_features, hidden_dim=16
#             )

#         # Категориальные main‑эффекты
#         self.categorical_main_effect = CategoricalMainEffects(vocabulary)

#         # Токенизация для трансформера
#         self.continuous_token_encoder = ContinuousTokenEncoder(
#             num_continuous_features, embedding_dim
#         )
#         self.categorical_embedding = ColumnEmbedding(vocabulary, embedding_dim)

#         # Общие колоночные эмбеддинги для всех признаков
#         total_features = num_continuous_features + self.num_categorical_features
#         self.column_embedding = nn.Parameter(torch.randn(total_features, embedding_dim) * 0.02)

#         # Трансформер (если используется)
#         if use_transformer:
#             encoder_layer = nn.TransformerEncoderLayer(
#                 d_model=embedding_dim,
#                 nhead=nhead,
#                 dim_feedforward=dim_feedforward,
#                 dropout=dropout,
#                 activation="gelu",
#                 batch_first=True,
#                 norm_first=True
#             )
#             self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers,
#                                                       norm=nn.LayerNorm(embedding_dim))
#             self.interaction_head = nn.Sequential(
#                 nn.Linear(embedding_dim, embedding_dim),
#                 nn.GELU(),
#                 nn.Dropout(dropout),
#                 nn.Linear(embedding_dim, output_dim)
#             )
#         else:
#             self.transformer = None
#             self.interaction_head = None

#         # Голова для main‑эффектов
#         self.main_head = nn.Linear(total_features, output_dim, bias=False)

#     def forward(self, categorical_x, continuous_x, return_explanations=False):
#         batch_size = continuous_x.size(0)

#         # Нормализация непрерывных
#         continuous_x_norm = self.continuous_norm(continuous_x)

#         # Main‑эффекты
#         continuous_effects = self.continuous_main_effect(continuous_x_norm)
#         if self.num_categorical_features > 0:
#             categorical_effects = self.categorical_main_effect(categorical_x)
#         else:
#             categorical_effects = continuous_x.new_zeros(batch_size, 0)

#         all_effects = torch.cat([continuous_effects, categorical_effects], dim=1)
#         main_prediction = self.main_head(all_effects)

#         # Если трансформер не используется, возвращаем только main
#         if not self.use_transformer:
#             if return_explanations:
#                 return {
#                     "prediction": main_prediction,
#                     "continuous_effects": continuous_effects,
#                     "categorical_effects": categorical_effects,
#                     "all_effects": all_effects,
#                     "main_prediction": main_prediction,
#                     "interaction_prediction": torch.zeros_like(main_prediction)
#                 }
#             return main_prediction

#         # Токены для трансформера
#         continuous_tokens = self.continuous_token_encoder(continuous_x_norm)
#         cat_tokens_list = []
#         for i, col in enumerate(self.vocabulary):
#             token = self.categorical_embedding(categorical_x[:, i], col)
#             cat_tokens_list.append(token.unsqueeze(1))
#         if len(cat_tokens_list) > 0:
#             categorical_tokens = torch.cat(cat_tokens_list, dim=1)
#             tokens = torch.cat([categorical_tokens, continuous_tokens], dim=1)
#         else:
#             tokens = continuous_tokens

#         # Добавляем колоночные эмбеддинги
#         tokens = tokens + self.column_embedding.unsqueeze(0)

#         # Трансформер
#         transformed = self.transformer(tokens)
#         pooled = transformed.mean(dim=1)
#         interaction_prediction = self.interaction_head(pooled)

#         # Финальный прогноз
#         prediction = main_prediction + interaction_prediction

#         if return_explanations:
#             return {
#                 "prediction": prediction,
#                 "continuous_effects": continuous_effects,
#                 "categorical_effects": categorical_effects,
#                 "all_effects": all_effects,
#                 "main_prediction": main_prediction,
#                 "interaction_prediction": interaction_prediction,
#                 "transformer_tokens": transformed
#             }
#         return prediction

In [7]:
def get_dataset_config(dataset_name):
    """Возвращает конфигурацию для заданного датасета (аналог DATASET_CONFIG)."""
    # Предполагаем, что у вас уже определён словарь DATASET_CONFIG.
    # Если нет – задайте его вручную.
    # Здесь для примера возвращаем конфигурацию для multi_seg.
    configs = {
         'bankmarketing': {
        'folder': 'bankmarketing',
        'target': 'Output',
        'continuous': ['age', 'balance', 'duration', 'campaign', 'pdays', 'previous'],
        'categorical': ['job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'day', 'month', 'poutcome'],
        'task': 'binary',
        'output_dim': 2,
        'metric': 'auc'
    },
    'onlineshoper': {
        'folder': 'onlineshoper',
        'target': 'Revenue',
        'continuous': ['Administrative_Duration', 'Informational_Duration', 'ProductRelated_Duration', 'BounceRates', 'ExitRates', 'PageValues'],
        'categorical': ['Administrative', 'Informational', 'ProductRelated', 'SpecialDay', 'Month', 'OperatingSystems', 'Browser', 'Region', 'TrafficType', 'VisitorType', 'Weekend'],
        'task': 'binary',
        'output_dim': 2,
        'metric': 'auc'
    },
    'cahouse': {
        'folder': 'cahouse',
        'target': 'Output',
        'continuous': ['longitude', 'latitude', 'housing_median_age', 'total_rooms', 'total_bedrooms', 'population', 'households', 'median_income'],
        'categorical': ['ocean_proximity'],
        'task': 'regression',
        'output_dim': 1,
        'metric': 'rmse'
    },
    'multi_forest': {
        'folder': 'multi_forest',
        'target': 'class',
        'continuous': ['elevation', 'aspect', 'slope', 'horizontal_distance_to_hydrology', 'Vertical_Distance_To_Hydrology', 'Horizontal_Distance_To_Roadways', 'Hillshade_9am', 'Hillshade_Noon', 'Hillshade_3pm', 'Horizontal_Distance_To_Fire_Points'],
        'categorical': ['wilderness_area1', 'wilderness_area2', 'wilderness_area3', 'wilderness_area4', 'soil_type_1', 'soil_type_2', 'soil_type_3', 'soil_type_4', 'soil_type_5', 'soil_type_6', 'soil_type_7', 'soil_type_8', 'soil_type_9', 'soil_type_10', 'soil_type_11', 'soil_type_12', 'soil_type_13', 'soil_type_14', 'soil_type_15', 'soil_type_16', 'soil_type_17', 'soil_type_18', 'soil_type_19', 'soil_type_20', 'soil_type_21', 'soil_type_22', 'soil_type_23', 'soil_type_24', 'soil_type_25', 'soil_type_26', 'soil_type_27', 'soil_type_28', 'soil_type_29', 'soil_type_30', 'soil_type_31', 'soil_type_32', 'soil_type_33', 'soil_type_34', 'soil_type_35', 'soil_type_36', 'soil_type_37', 'soil_type_38', 'soil_type_39', 'soil_type_40'],
        'task': 'multiclass',
        'output_dim': 7,
        'metric': 'f1'
    },
    'multi_seg': {
        'folder': 'multi_seg',
        'target': 'class',
        'continuous': ['region.centroid.col', 'region.centroid.row', 'region.pixel.count', 'short.line.density.5', 'short.line.density.2', 'vedge.mean', 'vegde.sd', 'hedge.mean', 'hedge.sd', 'intensity.mean', 'rawred.mean', 'rawblue.mean', 'rawgreen.mean', 'exred.mean', 'exblue.mean', 'exgreen.mean', 'value.mean', 'saturation.mean', 'hue.mean'],
        'categorical': [],
        'task': 'multiclass',
        'output_dim': 7,
        'metric': 'f1'
    },
    'cpu_small': {
        'folder': 'cpu_small',
        'target': 'usr',
        'continuous': ['lread', 'lwrite', 'scall', 'sread', 'swrite', 'fork', 'exec', 'rchar', 'wchar', 'runqsz', 'freemem', 'freeswap'],
        'categorical': [],
        'task': 'regression',
        'output_dim': 1,
        'metric': 'rmse'
    },
    'sarcos': {
        'folder': 'sarcos',
        'target': 'V22',
        'continuous': ['V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'V10', 'V11', 'V12', 'V13', 'V14', 'V15', 'V16', 'V17', 'V18', 'V19', 'V20', 'V21', 'V23', 'V24', 'V25', 'V26', 'V27', 'V28'],
        'categorical': [],
        'task': 'regression',
        'output_dim': 1,
        'metric': 'rmse'
    }
    }
    return configs[dataset_name]

In [8]:
def load_fold_data(dataset_name, fold, base_path="templates"):
    config = get_dataset_config(dataset_name)
    folder = config["folder"]
    target = config["target"]

    train_df = pd.read_csv(Path(base_path) / folder / f"Fold{fold}" / "train.csv").fillna("EMPTY")
    val_df   = pd.read_csv(Path(base_path) / folder / f"Fold{fold}" / "val.csv").fillna("EMPTY")
    test_df  = pd.read_csv(Path(base_path) / folder / f"Fold{fold}" / "test.csv").fillna("EMPTY")

    # Кодируем target
    if train_df[target].dtype == object:
        le = LabelEncoder()
        all_targets = pd.concat([train_df[target], val_df[target], test_df[target]])
        le.fit(all_targets)
        train_df[target] = le.transform(train_df[target])
        val_df[target]   = le.transform(val_df[target])
        test_df[target]  = le.transform(test_df[target])

    cat_cols = config["categorical"]
    cont_cols = config["continuous"]

    # Строим vocabulary на основе всех данных (train+val+test)
    vocabulary = {}
    if cat_cols:
        for col in cat_cols:
            all_vals = pd.concat([train_df[col], val_df[col], test_df[col]]).unique()
            all_vals = np.sort(all_vals.astype(str))  # приводим к строке для единообразия
            vocabulary[col] = {str(v): i for i, v in enumerate(all_vals)}

    # Кодируем категориальные с помощью маппинга vocabulary
    def encode_cat(df, vocab):
        encoded = np.zeros((len(df), len(vocab)), dtype=np.int64)
        for i, col in enumerate(vocab.keys()):
            # Используем .map с заменой неизвестных на 0 (можно использовать -1, но тогда нужно избегать отрицательных индексов)
            encoded[:, i] = df[col].astype(str).map(vocab[col]).fillna(0).values
        return encoded

    if cat_cols:
        X_train_cat = encode_cat(train_df, vocabulary)
        X_val_cat   = encode_cat(val_df, vocabulary)
        X_test_cat  = encode_cat(test_df, vocabulary)
    else:
        X_train_cat = np.empty((len(train_df), 0), dtype=np.int64)
        X_val_cat   = np.empty((len(val_df), 0), dtype=np.int64)
        X_test_cat  = np.empty((len(test_df), 0), dtype=np.int64)

    X_train_cont = train_df[cont_cols].values.astype(np.float32)
    X_val_cont   = val_df[cont_cols].values.astype(np.float32)
    X_test_cont  = test_df[cont_cols].values.astype(np.float32)

    return {
        "X_train_cat": X_train_cat,
        "X_train_cont": X_train_cont,
        "y_train": train_df[target].values,
        "X_val_cat": X_val_cat,
        "X_val_cont": X_val_cont,
        "y_val": val_df[target].values,
        "X_test_cat": X_test_cat,
        "X_test_cont": X_test_cont,
        "y_test": test_df[target].values,
        "vocabulary": vocabulary,
        "num_continuous": len(cont_cols),
        "output_dim": config["output_dim"],
        "task": config["task"]
    }

In [9]:
def train_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0
    for batch in loader:
        cat, cont, y = batch
        cat = cat.to(device)
        cont = cont.to(device)
        y = y.to(device).float() if model.output_dim == 1 else y.to(device).long()
        optimizer.zero_grad()
        pred = model(cat, cont)
        if model.output_dim == 1:
            loss = criterion(pred.squeeze(), y)
        else:
            loss = criterion(pred, y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)


def evaluate_model(model, loader, device, task="multiclass"):
    model.eval()
    preds_list = []
    targets_list = []
    with torch.no_grad():
        for batch in loader:
            cat, cont, y = batch
            cat = cat.to(device)
            cont = cont.to(device)
            pred = model(cat, cont)
            if task == "binary":
                if pred.size(1) == 1:
                    pred = pred.squeeze()
                else:
                    pred = pred[:, 1]  # вероятности для положительного класса
                preds_list.append(pred.cpu().numpy())
            elif task == "multiclass":
                preds_list.append(torch.argmax(pred, dim=1).cpu().numpy())
            else:  # regression
                preds_list.append(pred.squeeze().cpu().numpy())
            targets_list.append(y.cpu().numpy())
    all_preds = np.concatenate(preds_list)
    all_targets = np.concatenate(targets_list)

    if task == "binary":
        # используем ROC-AUC
        if len(np.unique(all_targets)) == 2:
            return roc_auc_score(all_targets, all_preds)
        else:
            return accuracy_score(all_targets, (all_preds > 0.5).astype(int))
    elif task == "multiclass":
        return f1_score(all_targets, all_preds, average="macro")
    else:
        return np.sqrt(mean_squared_error(all_targets, all_preds))

In [10]:
from tqdm import trange
import torch.optim.lr_scheduler as lr_scheduler

def train_and_evaluate_model(
    model,
    train_data,
    val_data,
    test_data,
    batch_size=256,
    epochs=100,
    lr=1e-3,
    weight_decay=1e-5,
    early_stopping_patience=50,
    device="cpu",
    metric_mode='max'          # 'max' или 'min'
):
    # Создаём DataLoader'ы
    train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_data, batch_size=batch_size, shuffle=False)
    test_loader = DataLoader(test_data, batch_size=batch_size, shuffle=False)

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    if model.output_dim == 1:
        criterion = nn.MSELoss() if model.task == "regression" else nn.BCEWithLogitsLoss()
    else:
        criterion = nn.CrossEntropyLoss()

    # Добавляем планировщик
    scheduler = lr_scheduler.ReduceLROnPlateau(
        optimizer, mode=metric_mode, factor=0.5, patience=3
    )

    best_val_metric = -np.inf if metric_mode == 'max' else np.inf
    patience_counter = 0
    best_model_state = None

    for epoch in trange(epochs):
        train_loss = train_epoch(model, train_loader, optimizer, criterion, device)
        val_metric = evaluate_model(model, val_loader, device, task=model.task)

        # Обновляем scheduler на основе валидационной метрики
        scheduler.step(val_metric)

        # Сохраняем лучшую модель
        is_better = (metric_mode == 'max' and val_metric > best_val_metric) or \
                    (metric_mode == 'min' and val_metric < best_val_metric)
        if is_better:
            best_val_metric = val_metric
            patience_counter = 0
            best_model_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        else:
            patience_counter += 1
            if patience_counter >= early_stopping_patience:
                print(f"Early stopping at epoch {epoch+1}")
                break

    # Восстанавливаем лучшую модель
    model.load_state_dict(best_model_state)
    model.to(device)

    # Оценка на тесте
    test_metric = evaluate_model(model, test_loader, device, task=model.task)
    return test_metric, best_val_metric

In [11]:
import os
import torch
import pandas as pd
import numpy as np
from pathlib import Path
import gc

# Предполагается, что у вас уже есть все классы и функции из предыдущих блоков (KAN, модели, load_fold_data, train_and_evaluate_model и т.д.)
# Для краткости мы их не повторяем – они должны быть определены выше.

def run_kan_experiment_with_save(
    dataset_name: str,
    model_type: str,          # "kan" или "mlp"
    use_transformer: bool,
    epochs: int = 80,
    batch_size: int = 256,
    lr: float = 1e-3,
    weight_decay: float = 1e-5,
    early_stopping_patience: int = 15,
    device: str = "cpu",
    save_dir: str = "models"
):
    """
    Обучает модель на всех фолдах, сохраняет каждую модель и возвращает DataFrame с метриками.
    """
    variant = f"{model_type}_transformer" if use_transformer else f"{model_type}_additive"
    print(f"\n{'='*60}")
    print(f"Running experiment: {dataset_name} / {variant}")
    print(f"{'='*60}")

    # Создаём папку для сохранения моделей
    model_save_path = Path(save_dir) / dataset_name / variant
    model_save_path.mkdir(parents=True, exist_ok=True)

    all_results = []

    for fold in range(1, 6):
        print(f"  Fold {fold}...")
        # Загружаем данные для фолда
        data = load_fold_data(dataset_name, fold)

        # Подготовка тензоров
        X_train_cat = torch.tensor(data["X_train_cat"], dtype=torch.long)
        X_train_cont = torch.tensor(data["X_train_cont"], dtype=torch.float32)
        y_train = torch.tensor(data["y_train"], dtype=torch.long if data["task"] == "multiclass" else torch.float32)
        X_val_cat = torch.tensor(data["X_val_cat"], dtype=torch.long)
        X_val_cont = torch.tensor(data["X_val_cont"], dtype=torch.float32)
        y_val = torch.tensor(data["y_val"], dtype=torch.long if data["task"] == "multiclass" else torch.float32)
        X_test_cat = torch.tensor(data["X_test_cat"], dtype=torch.long)
        X_test_cont = torch.tensor(data["X_test_cont"], dtype=torch.float32)
        y_test = torch.tensor(data["y_test"], dtype=torch.long if data["task"] == "multiclass" else torch.float32)

        train_data = TensorDataset(X_train_cat, X_train_cont, y_train)
        val_data   = TensorDataset(X_val_cat, X_val_cont, y_val)
        test_data  = TensorDataset(X_test_cat, X_test_cont, y_test)

        # Создаём модель
        output_dim = 1 if data["task"] in ["binary", "regression"] else data["output_dim"]
        model = InterpretableTabTransformer(
            output_dim=output_dim,
            vocabulary=data["vocabulary"],
            num_continuous_features=data["num_continuous"],
            embedding_dim=64,
            nhead=8,
            num_layers=3,
            dim_feedforward=128,
            dropout=0.1,
            main_effect_type=model_type,
            kan_hidden_dim=16,
            use_transformer=use_transformer
        )
        model.task = data["task"]
        model.to(device)

        start = time.time()
        test_metric, val_metric = train_and_evaluate_model(
            model,
            train_data,
            val_data,
            test_data,
            batch_size=batch_size,
            epochs=epochs,
            lr=lr,
            weight_decay=weight_decay,
            early_stopping_patience=early_stopping_patience,
            device=device
        )
        train_time = time.time() - start

        # Сохраняем модель
        model_path = model_save_path / f"fold{fold}_3.pt"
        torch.save(model.state_dict(), model_path)
        print(f"    Model saved to {model_path}")

        # Считаем количество параметров
        num_params = sum(p.numel() for p in model.parameters())
        print(f"    Fold {fold} test metric: {test_metric:.4f}, val metric: {val_metric:.4f}, time: {train_time:.2f}s")

        all_results.append({
            "dataset": dataset_name,
            "fold": fold,
            "variant": variant,
            "model_type": model_type,
            "use_transformer": use_transformer,
            "metric": test_metric,
            "val_metric": val_metric,
            "time": train_time,
            "params": num_params,
            "model_path": str(model_path)
        })

        # Очистка памяти
        del model, train_data, val_data, test_data
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    return pd.DataFrame(all_results)


In [49]:
# Список датасетов (можно менять)
datasets_to_run = [
    ['bankmarketing', 1e-3],
    ['onlineshoper', 1e-3],
    ['multi_seg', 1e-3],
    ['multi_forest', 1e-3],
    ['cahouse', 5e-2],
    ['cpu_small', 5e-2],
    ['sarcos', 5e-2]
]
# Варианты моделей
model_variants = [
    {"model_type": "kan", "use_transformer": False, "label": "KAN_additive"},
    {"model_type": "mlp", "use_transformer": False, "label": "MLP_additive"},
    {"model_type": "kan", "use_transformer": True,  "label": "KAN_transformer"},
    {"model_type": "mlp", "use_transformer": True,  "label": "MLP_transformer"},
]

all_dfs = []

for ds, lr in datasets_to_run:
    for variant in model_variants:
        df = run_kan_experiment_with_save(
            dataset_name=ds,
            model_type=variant["model_type"],
            use_transformer=variant["use_transformer"],
            epochs=100,          # можно увеличить
            lr = lr,
            device=device,
            save_dir="models_saved_2.0"
        )
        # Добавляем столбец с меткой
        df["variant"] = variant["label"]
        all_dfs.append(df)

# Объединяем все результаты
final_df = pd.concat(all_dfs, ignore_index=True)
final_df.to_csv("kan_mlp_experiment_results_all_5.0.csv", index=False)
print("\nAll experiments finished. Results saved to kan_mlp_experiment_results.csv")


Running experiment: bankmarketing / kan_additive
  Fold 1...


 67%|██████████████████████████████████████████████████████▎                          | 67/100 [02:46<01:22,  2.49s/it]

Early stopping at epoch 68


    Model saved to models_saved_2.0\bankmarketing\kan_additive\fold1_3.pt
    Fold 1 test metric: 0.9179, val metric: 0.9156, time: 167.16s
  Fold 2...


 77%|██████████████████████████████████████████████████████████████▎                  | 77/100 [03:08<00:56,  2.45s/it]

...

Early stopping at epoch 18


    Model saved to models_saved_2.0\sarcos\mlp_transformer\fold4_3.pt
    Fold 4 test metric: 5.5096, val metric: 5.5304, time: 63.92s
  Fold 5...


C:\Users\user\anaconda3\envs\rtx_env\lib\site-packages\torch\nn\modules\transformer.py:379: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(
 15%|████████████▏                                                                    | 15/100 [00:55<05:16,  3.73s/it]

Early stopping at epoch 16


    Model saved to models_saved_2.0\sarcos\mlp_transformer\fold5_3.pt
    Fold 5 test metric: 7.0090, val metric: 6.9930, time: 56.22s

All experiments finished. Results saved to kan_mlp_experiment_results.csv


In [50]:
print(final_df.groupby(['dataset', 'variant'])['metric'].mean())
print(final_df.groupby(['dataset', 'variant'])['metric'].std()) #3 слоя

dataset        variant        
bankmarketing  KAN_additive        0.915960
               KAN_transformer     0.933511
               MLP_additive        0.915192
               MLP_transformer     0.932102
cahouse        KAN_additive        7.360409
               KAN_transformer     7.357440
               MLP_additive        7.502016
               MLP_transformer     7.159930
cpu_small      KAN_additive       16.003374
               KAN_transformer    17.174952
               MLP_additive       13.009436
               MLP_transformer    13.891908
multi_forest   KAN_additive        0.441698
               KAN_transformer     0.719229
               MLP_additive        0.427679
               MLP_transformer     0.721165
multi_seg      KAN_additive        0.852508
               KAN_transformer     0.970459
               MLP_additive        0.937581
               MLP_transformer     0.967979
onlineshoper   KAN_additive        0.912433
               KAN_transformer     0.925592
 

## Интерпретация

In [12]:
class FeatureWiseKANEncoder(nn.Module):
    def __init__(self, num_features, embedding_dim, hidden_dim=16, grid_size=5, spline_order=3):
        super().__init__()
        self.models = nn.ModuleList([
            KAN([1, hidden_dim, embedding_dim], grid_size=grid_size, spline_order=spline_order)
            for _ in range(num_features)
        ])

    def forward(self, x):
        # x: (batch, num_features)
        tokens = []
        for i, model in enumerate(self.models):
            xi = x[:, i:i+1]  # (batch, 1)
            token = model(xi)  # (batch, embedding_dim)
            tokens.append(token.unsqueeze(1))
        return torch.cat(tokens, dim=1)

In [13]:
class InterpretableTabTransformerV2(nn.Module):
    def __init__(
        self,
        output_dim,
        vocabulary: Dict[str, Dict[str, int]],
        num_continuous_features: int,
        embedding_dim: int = 64,
        nhead: int = 8,
        num_layers: int = 3,
        dim_feedforward: int = 128,
        dropout: float = 0.1,
        kan_hidden_dim: int = 16,
        grid_size: int = 5,
        spline_order: int = 3,
        use_transformer: bool = True  # оставлено для совместимости
    ):
        super().__init__()
        self.output_dim = output_dim
        self.vocabulary = vocabulary
        self.num_continuous_features = num_continuous_features
        self.embedding_dim = embedding_dim
        self.num_categorical_features = len(vocabulary)

        # Нормализация непрерывных
        self.continuous_norm = nn.BatchNorm1d(num_continuous_features)

        # Per‑feature KAN → эмбеддинги размерности embedding_dim
        self.continuous_token_encoder = FeatureWiseKANEncoder(
            num_features=num_continuous_features,
            embedding_dim=embedding_dim,
            hidden_dim=kan_hidden_dim,
            grid_size=grid_size,
            spline_order=spline_order
        )

        # Категориальные эмбеддинги
        from tabkanet.models.base_blocks import ColumnEmbedding  # предположим, что он доступен
        self.categorical_embedding = ColumnEmbedding(vocabulary, embedding_dim)

        # Колоночные эмбеддинги
        total_features = num_continuous_features + self.num_categorical_features
        self.column_embedding = nn.Parameter(torch.randn(total_features, embedding_dim) * 0.02)

        # Трансформер
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embedding_dim,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            activation='gelu',
            batch_first=True,
            norm_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers,
                                                  norm=nn.LayerNorm(embedding_dim))

        # MLP-голова (как в TabKANet)
        self.classifier = nn.Sequential(
            nn.Linear(embedding_dim, embedding_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(embedding_dim, output_dim)
        )

    def forward(self, categorical_x, continuous_x, return_explanations=False):
        batch_size = continuous_x.size(0)

        continuous_x_norm = self.continuous_norm(continuous_x)

        # Непрерывные токены (через KAN)
        continuous_tokens = self.continuous_token_encoder(continuous_x_norm)  # (batch, num_cont, embedding_dim)

        # Категориальные токены
        cat_tokens_list = []
        for i, col in enumerate(self.vocabulary):
            token = self.categorical_embedding(categorical_x[:, i], col)
            cat_tokens_list.append(token.unsqueeze(1))
        if len(cat_tokens_list) > 0:
            categorical_tokens = torch.cat(cat_tokens_list, dim=1)
            tokens = torch.cat([categorical_tokens, continuous_tokens], dim=1)
        else:
            tokens = continuous_tokens

        # Добавляем колоночные эмбеддинги
        tokens = tokens + self.column_embedding.unsqueeze(0)

        # Трансформер
        transformed = self.transformer(tokens)
        pooled = transformed.mean(dim=1)
        prediction = self.classifier(pooled)

        if return_explanations:
            return {
                "prediction": prediction,
                "continuous_tokens": continuous_tokens,
                "categorical_tokens": categorical_tokens if len(cat_tokens_list) > 0 else None,
                "transformer_output": transformed,
                "pooled": pooled
            }
        return prediction

In [14]:
def run_interpretable_v2_experiment(
    dataset_name: str,
    epochs: int = 80,
    batch_size: int = 256,
    lr: float = 1e-3,
    weight_decay: float = 1e-5,
    early_stopping_patience: int = 10,
    device: str = "cpu",
    save_dir: str = "models_v2"
):
    """
    Запускает эксперимент с новой интерпретируемой моделью (InterpretableTabTransformerV2)
    на всех фолдах и сохраняет модели.
    """
    variant = "interpretable_v2"
    print(f"\n{'='*60}")
    print(f"Running experiment: {dataset_name} / {variant}")
    print(f"{'='*60}")

    model_save_path = Path(save_dir) / dataset_name / variant
    model_save_path.mkdir(parents=True, exist_ok=True)

    all_results = []

    for fold in range(1, 6):
        print(f"  Fold {fold}...")
        data = load_fold_data(dataset_name, fold)  # предполагается, что эта функция существует

        # Подготовка тензоров
        X_train_cat = torch.tensor(data["X_train_cat"], dtype=torch.long)
        X_train_cont = torch.tensor(data["X_train_cont"], dtype=torch.float32)
        y_train = torch.tensor(data["y_train"], dtype=torch.long if data["task"] == "multiclass" else torch.float32)
        X_val_cat = torch.tensor(data["X_val_cat"], dtype=torch.long)
        X_val_cont = torch.tensor(data["X_val_cont"], dtype=torch.float32)
        y_val = torch.tensor(data["y_val"], dtype=torch.long if data["task"] == "multiclass" else torch.float32)
        X_test_cat = torch.tensor(data["X_test_cat"], dtype=torch.long)
        X_test_cont = torch.tensor(data["X_test_cont"], dtype=torch.float32)
        y_test = torch.tensor(data["y_test"], dtype=torch.long if data["task"] == "multiclass" else torch.float32)

        train_data = TensorDataset(X_train_cat, X_train_cont, y_train)
        val_data   = TensorDataset(X_val_cat, X_val_cont, y_val)
        test_data  = TensorDataset(X_test_cat, X_test_cont, y_test)

        # Определяем output_dim
        if data["task"] in ["binary", "multiclass"]:
            output_dim = data["output_dim"] if data["task"] == "multiclass" else 2
        else:
            output_dim = 1

        model = InterpretableTabTransformerV2(
            output_dim=output_dim,
            vocabulary=data["vocabulary"],
            num_continuous_features=data["num_continuous"],
            embedding_dim=64,
            nhead=8,
            num_layers=3,
            dim_feedforward=128,
            dropout=0.1,
            kan_hidden_dim=16
        )
        model.task = data["task"]
        model.to(device)

        start = time.time()
        test_metric, val_metric = train_and_evaluate_model(
            model,
            train_data,
            val_data,
            test_data,
            batch_size=batch_size,
            epochs=epochs,
            lr=lr,
            weight_decay=weight_decay,
            early_stopping_patience=early_stopping_patience,
            device=device,
        )
        train_time = time.time() - start

        model_path = model_save_path / f"fold{fold}_v2.pt"
        torch.save(model.state_dict(), model_path)
        print(f"    Model saved to {model_path}")

        num_params = sum(p.numel() for p in model.parameters())
        print(f"    Fold {fold} test metric: {test_metric:.4f}, val metric: {val_metric:.4f}, time: {train_time:.2f}s")

        all_results.append({
            "dataset": dataset_name,
            "fold": fold,
            "variant": variant,
            "metric": test_metric,
            "val_metric": val_metric,
            "time": train_time,
            "params": num_params,
            "model_path": str(model_path)
        })

        del model, train_data, val_data, test_data
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    return pd.DataFrame(all_results)

In [17]:
datasets = datasets_to_run = [
    ['bankmarketing', 1e-3],
    ['onlineshoper', 1e-3],
    ['multi_seg', 1e-3],
    ['multi_forest', 1e-3],
    ['cahouse', 5e-2],
    ['cpu_small', 5e-2],
    ['sarcos', 5e-2]
]
all_results_v2 = []
for ds, lr in datasets:
    df = run_interpretable_v2_experiment(ds, lr = lr, device='cuda' if torch.cuda.is_available() else 'cpu')
    all_results_v2.append(df)
final_df_v2 = pd.concat(all_results_v2, ignore_index=True)
final_df_v2.to_csv("results_interpretable_v2.csv", index=False)


Running experiment: bankmarketing / interpretable_v2
  Fold 1...


C:\Users\user\anaconda3\envs\rtx_env\lib\site-packages\torch\nn\modules\transformer.py:379: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(
 28%|██████████████████████▌                                                           | 22/80 [01:10<03:05,  3.19s/it]

Early stopping at epoch 23


    Model saved to models_v2\bankmarketing\interpretable_v2\fold1_v2.pt
    Fold 1 test metric: 0.9318, val metric: 0.9316, time: 71.81s
  Fold 2...


C:\Users\user\anaconda3\envs\rtx_env\lib\site-packages\torch\nn\modules\transformer.py:379: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(
 25%|████████████████████▌                                                             | 20/80 [01:03<03:09,  3.15s/it]

...

Early stopping at epoch 11


    Model saved to models_v2\sarcos\interpretable_v2\fold4_v2.pt
    Fold 4 test metric: 17.8901, val metric: 17.6351, time: 129.54s
  Fold 5...


C:\Users\user\anaconda3\envs\rtx_env\lib\site-packages\torch\nn\modules\transformer.py:379: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(
 14%|███████████▎                                                                      | 11/80 [02:19<14:38, 12.73s/it]

Early stopping at epoch 12


    Model saved to models_v2\sarcos\interpretable_v2\fold5_v2.pt
    Fold 5 test metric: 9.5199, val metric: 9.5497, time: 141.39s


In [18]:
print(final_df_v2.groupby(['dataset', 'variant'])['metric'].mean())
print(final_df_v2.groupby(['dataset', 'variant'])['metric'].std()) 

dataset        variant         
bankmarketing  interpretable_v2     0.931508
cahouse        interpretable_v2     8.466110
cpu_small      interpretable_v2    15.099198
multi_forest   interpretable_v2     0.725463
multi_seg      interpretable_v2     0.969711
onlineshoper   interpretable_v2     0.926809
sarcos         interpretable_v2    13.314909
Name: metric, dtype: float64
dataset        variant         
bankmarketing  interpretable_v2    0.001640
cahouse        interpretable_v2    0.410062
cpu_small      interpretable_v2    3.954611
multi_forest   interpretable_v2    0.002876
multi_seg      interpretable_v2    0.009260
onlineshoper   interpretable_v2    0.003913
sarcos         interpretable_v2    4.225554
Name: metric, dtype: float64


In [34]:
print(final_df_v2.groupby(['dataset', 'variant'])['metric'].mean())
print(final_df_v2.groupby(['dataset', 'variant'])['metric'].std()) 

dataset        variant         
bankmarketing  interpretable_v2     0.932842
cahouse        interpretable_v2    12.295974
cpu_small      interpretable_v2    80.928295
multi_forest   interpretable_v2     0.723497
multi_seg      interpretable_v2     0.970336
onlineshoper   interpretable_v2     0.926139
Name: metric, dtype: float64
dataset        variant         
bankmarketing  interpretable_v2    0.000840
cahouse        interpretable_v2    0.449730
cpu_small      interpretable_v2    0.511530
multi_forest   interpretable_v2    0.005150
multi_seg      interpretable_v2    0.013527
onlineshoper   interpretable_v2    0.004417
Name: metric, dtype: float64


In [15]:
def ensure_dir(path):
    Path(path).mkdir(parents=True, exist_ok=True)

def safe_filename(name):
    return str(name).replace("/", "_").replace("\\", "_").replace(" ", "_").replace(":", "_")

In [16]:
def save_kan_functions(
    model,
    continuous_data,
    feature_names,
    output_dir,
    n_points=300,
    device="cuda",
):
    """
    LEVEL 1.
    Сохраняет непосредственно функцию: x_i -> KAN_i(x_i)
    Это НЕ SHAP. Это НЕ PDP. Это результат forward pass через KAN.
    """
    if model.main_effect_type != "kan":
        raise ValueError("save_kan_functions() requires a KAN model.")

    ensure_dir(output_dir)
    model.eval()
    continuous_data = np.asarray(continuous_data, dtype=np.float32)
    kan_models = model.continuous_main_effect.models

    for feature_index, feature_name in enumerate(feature_names):
        raw_values = np.linspace(
            continuous_data[:, feature_index].min(),
            continuous_data[:, feature_index].max(),
            n_points
        )
        raw_tensor = torch.tensor(raw_values, dtype=torch.float32, device=device)
        dummy = torch.zeros(n_points, model.num_continuous_features, device=device)
        dummy[:, feature_index] = raw_tensor

        with torch.no_grad():
            normalized = model.continuous_norm(dummy)
            normalized_feature = normalized[:, feature_index].reshape(-1, 1)
            kan_output = kan_models[feature_index](normalized_feature)

        kan_output = kan_output.detach().cpu().numpy().reshape(-1)
        normalized_values = normalized_feature.detach().cpu().numpy().reshape(-1)

        df = pd.DataFrame({
            "feature_value": raw_values,
            "normalized_value": normalized_values,
            "kan_function": kan_output,
        })
        df.to_csv(os.path.join(output_dir, safe_filename(feature_name) + "_KAN_function.csv"), index=False)

        # Сохраняем внутренние параметры сплайнов
        feature_dir = os.path.join(output_dir, safe_filename(feature_name) + "_KAN_internal")
        ensure_dir(feature_dir)
        kan = kan_models[feature_index]
        for layer_index, layer in enumerate(kan.layers):
            np.save(os.path.join(feature_dir, f"layer_{layer_index}_grid.npy"), layer.grid.detach().cpu().numpy())
            np.save(os.path.join(feature_dir, f"layer_{layer_index}_spline_weight.npy"), layer.spline_weight.detach().cpu().numpy())
            if hasattr(layer, "spline_scaler"):
                np.save(os.path.join(feature_dir, f"layer_{layer_index}_spline_scaler.npy"), layer.spline_scaler.detach().cpu().numpy())
            np.save(os.path.join(feature_dir, f"layer_{layer_index}_base_weight.npy"), layer.base_weight.detach().cpu().numpy())
        print("Saved KAN function:", feature_name)


In [17]:
def save_mlp_feature_curves(
    model,
    continuous_data,
    categorical_data,
    feature_names,
    output_dir,
    n_points=300,
    device="cuda",
):
    """
    Для MLP нет spline-функции. Сохраняем MLP_i(x_i) при фиксированных остальных.
    """
    if model.main_effect_type != "mlp":
        raise ValueError("This function is intended for an MLP model.")

    ensure_dir(output_dir)
    model.eval()
    continuous_data = np.asarray(continuous_data, dtype=np.float32)
    reference = np.median(continuous_data, axis=0)

    for feature_index, feature_name in enumerate(feature_names):
        x_values = np.linspace(
            continuous_data[:, feature_index].min(),
            continuous_data[:, feature_index].max(),
            n_points
        )
        effects = []
        for value in x_values:
            sample = reference.copy()
            sample[feature_index] = value
            sample = torch.tensor(sample, dtype=torch.float32, device=device).unsqueeze(0)
            with torch.no_grad():
                normalized = model.continuous_norm(sample)
                effect = model.continuous_main_effect(normalized)[0, feature_index]
            effects.append(effect.item())

        df = pd.DataFrame({
            "feature_value": x_values,
            "mlp_main_function": effects,
        })
        df.to_csv(os.path.join(output_dir, safe_filename(feature_name) + "_MLP_function.csv"), index=False)


In [18]:
def save_main_effects(
    model,
    categorical_x,
    continuous_x,
    continuous_names,
    categorical_names,
    output_path,
    device="cuda",
):
    model.eval()
    categorical_x = categorical_x.to(device)
    continuous_x = continuous_x.to(device)

    with torch.no_grad():
        result = model(categorical_x, continuous_x, return_explanations=True)

    continuous_effects = result["continuous_effects"].detach().cpu().numpy()
    categorical_effects = result["categorical_effects"].detach().cpu().numpy()
    n_samples = continuous_x.shape[0]

    rows = []
    for sample_id in range(n_samples):
        for feature_id, feature_name in enumerate(continuous_names):
            rows.append({
                "sample_id": sample_id,
                "feature": feature_name,
                "feature_type": "continuous",
                "value": continuous_x[sample_id, feature_id].detach().cpu().item(),
                "effect": continuous_effects[sample_id, feature_id],
            })
        for feature_id, feature_name in enumerate(categorical_names):
            rows.append({
                "sample_id": sample_id,
                "feature": feature_name,
                "feature_type": "categorical",
                "value": categorical_x[sample_id, feature_id].detach().cpu().item(),
                "effect": categorical_effects[sample_id, feature_id],
            })

    df = pd.DataFrame(rows)
    ensure_dir(os.path.dirname(output_path))
    df.to_csv(output_path, index=False)

    # Prediction decomposition
    prediction_df = pd.DataFrame({
        "sample_id": np.arange(n_samples),
        "main_prediction": result["main_prediction"].detach().cpu().numpy().reshape(-1),
        "interaction_prediction": result["interaction_prediction"].detach().cpu().numpy().reshape(-1),
        "prediction": result["prediction"].detach().cpu().numpy().reshape(-1),
    })
    prediction_path = output_path.replace(".csv", "_predictions.csv")
    prediction_df.to_csv(prediction_path, index=False)


In [19]:
def save_categorical_effects(
    model,
    category_names,
    category_decoders,
    output_dir,
    device="cuda",
):
    """Сохраняет эффекты для каждой категории."""
    ensure_dir(output_dir)
    model.eval()
    # Используем правильный атрибут – 'effects' (ModuleDict)
    effects = model.categorical_main_effect.effects

    for feature_name in category_names:
        embedding = effects[feature_name]
        weights = embedding.weight.detach().cpu().numpy().reshape(-1)
        decoder = category_decoders[feature_name]
        rows = []
        for category_id, effect in enumerate(weights):
            rows.append({
                "category_id": category_id,
                "category": decoder.get(category_id, str(category_id)),
                "effect": effect,
            })
        df = pd.DataFrame(rows)
        df.to_csv(os.path.join(output_dir, safe_filename(feature_name) + "_categorical_effects.csv"), index=False)

In [20]:
def save_shap_values(
    model,
    categorical_x,
    continuous_x,
    feature_names,
    output_dir,
    background_size=100,
    explain_size=200,
    device="cuda",
    target_output=0,
):
    try:
        import shap
    except ImportError:
        raise ImportError("Install SHAP: pip install shap")

    ensure_dir(output_dir)
    model.eval()

    continuous_np = continuous_x.detach().cpu().numpy()
    categorical_np = categorical_x.detach().cpu().numpy()
    n_cat = categorical_np.shape[1]
    n_cont = continuous_np.shape[1]
    X = np.concatenate([categorical_np, continuous_np], axis=1)
    all_feature_names = list(feature_names["categorical"]) + list(feature_names["continuous"])

    background_size = min(background_size, len(X))
    explain_size = min(explain_size, len(X))
    background = X[:background_size]
    X_explain = X[:explain_size]

    def predict_fn(X_numpy):
        X_numpy = np.asarray(X_numpy)
        cat = torch.tensor(X_numpy[:, :n_cat], dtype=torch.long, device=device)
        cont = torch.tensor(X_numpy[:, n_cat:], dtype=torch.float32, device=device)
        with torch.no_grad():
            prediction = model(cat, cont)
        prediction = prediction.detach().cpu().numpy()
        if prediction.ndim == 1:
            return prediction
        if prediction.shape[1] == 1:
            return prediction[:, 0]
        return prediction[:, target_output]

    masker = shap.maskers.Independent(background)
    explainer = shap.Explainer(predict_fn, masker, algorithm="permutation")
    shap_result = explainer(X_explain)
    shap_values = shap_result.values

    np.savez_compressed(
        os.path.join(output_dir, "shap_values.npz"),
        shap_values=shap_values,
        X=X_explain,
        feature_names=np.array(all_feature_names),
        base_value=np.asarray(shap_result.base_values),
    )

    rows = []
    for sample_id in range(len(X_explain)):
        for feature_id, feature_name in enumerate(all_feature_names):
            rows.append({
                "sample_id": sample_id,
                "feature": feature_name,
                "feature_type": "categorical" if feature_id < n_cat else "continuous",
                "feature_value": X_explain[sample_id, feature_id],
                "shap_value": shap_values[sample_id, feature_id],
            })
    df = pd.DataFrame(rows)
    df.to_csv(os.path.join(output_dir, "shap_values_long.csv"), index=False)

    importance = np.abs(shap_values).mean(axis=0)
    importance_df = pd.DataFrame({
        "feature": all_feature_names,
        "mean_abs_shap": importance,
    }).sort_values("mean_abs_shap", ascending=False).reset_index(drop=True)
    importance_df.to_csv(os.path.join(output_dir, "shap_global_importance.csv"), index=False)


In [21]:
def save_shap_feature_relationships(
    shap_csv_path,
    output_dir,
):
    ensure_dir(output_dir)
    df = pd.read_csv(shap_csv_path)
    continuous = df[df["feature_type"] == "continuous"]
    for feature in continuous["feature"].unique():
        feature_df = continuous[continuous["feature"] == feature][["sample_id", "feature_value", "shap_value"]]
        feature_df.to_csv(os.path.join(output_dir, safe_filename(feature) + "_SHAP_relationship.csv"), index=False)


In [22]:
def save_all_interpretability(
    model,
    categorical_x,
    continuous_x,
    continuous_names,
    categorical_names,
    category_decoders,
    output_dir,
    device="cuda",
    n_curve_points=300,
    shap_background_size=100,
    shap_explain_size=200,
):
    output_dir = Path(output_dir)
    ensure_dir(output_dir)

    # Level 1
    level1 = output_dir / "level_1_intrinsic"
    if model.main_effect_type == "kan":
        save_kan_functions(
            model=model,
            continuous_data=continuous_x.detach().cpu().numpy(),
            feature_names=continuous_names,
            output_dir=str(level1),
            n_points=n_curve_points,
            device=device,
        )
    else:
        save_mlp_feature_curves(
            model=model,
            continuous_data=continuous_x.detach().cpu().numpy(),
            categorical_data=categorical_x,
            feature_names=continuous_names,
            output_dir=str(level1),
            n_points=n_curve_points,
            device=device,
        )

    # Level 2
    level2 = output_dir / "level_2_main_effects"
    ensure_dir(level2)
    save_main_effects(
        model=model,
        categorical_x=categorical_x,
        continuous_x=continuous_x,
        continuous_names=continuous_names,
        categorical_names=categorical_names,
        output_path=str(level2 / "main_effects.csv"),
        device=device,
    )
    # Исправленный вызов – без cardinalities
    save_categorical_effects(
        model=model,
        category_names=categorical_names,
        category_decoders=category_decoders,
        output_dir=str(level2 / "categorical"),
        device=device,
    )

    # Level 3
    level3 = output_dir / "level_3_shap"
    ensure_dir(level3)
    save_shap_values(
        model=model,
        categorical_x=categorical_x,
        continuous_x=continuous_x,
        feature_names={"categorical": categorical_names, "continuous": continuous_names},
        output_dir=str(level3),
        background_size=shap_background_size,
        explain_size=shap_explain_size,
        device=device,
    )
    save_shap_feature_relationships(
        shap_csv_path=str(level3 / "shap_values_long.csv"),
        output_dir=str(level3 / "feature_relationships"),
    )

    print(f"\nInterpretation data saved to: {output_dir}")

In [23]:
def save_v2_interpretability(
    model,
    categorical_x,
    continuous_x,
    continuous_names,
    categorical_names,
    category_decoders,
    output_dir,
    device="cpu",
    n_points=300,
    shap_background_size=100,
    shap_explain_size=200,
):
    """
    Для модели V2: сохраняем кривые из continuous_tokens и SHAP.
    """
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    # ---- Level 1: кривые из первого канала непрерывных токенов ----
    level1 = output_dir / "level_1_tokens"
    level1.mkdir(parents=True, exist_ok=True)

    model.eval()
    continuous_data = continuous_x.cpu().numpy()
    device = next(model.parameters()).device

    for feature_index, feature_name in enumerate(continuous_names):
        x_values = np.linspace(
            continuous_data[:, feature_index].min(),
            continuous_data[:, feature_index].max(),
            n_points
        )
        effects = []
        reference = np.median(continuous_data, axis=0)
        for v in x_values:
            sample = reference.copy()
            sample[feature_index] = v
            cont_tensor = torch.tensor(sample, dtype=torch.float32, device=device).unsqueeze(0)
            cat_dummy = torch.zeros(1, categorical_x.shape[1], dtype=torch.long, device=device)
            with torch.no_grad():
                out = model(cat_dummy, cont_tensor, return_explanations=True)
                token = out["continuous_tokens"][0, feature_index, 0].item()
            effects.append(token)
        df = pd.DataFrame({
            "feature_value": x_values,
            "token_channel_0": effects
        })
        df.to_csv(level1 / f"{feature_name}_token_curve.csv", index=False)

    # ---- Level 3: SHAP ----
    level3 = output_dir / "level_3_shap"
    ensure_dir(level3)
    save_shap_values(
        model=model,
        categorical_x=categorical_x,
        continuous_x=continuous_x,
        feature_names={"categorical": categorical_names, "continuous": continuous_names},
        output_dir=str(level3),
        background_size=shap_background_size,
        explain_size=shap_explain_size,
        device=device,
    )
    save_shap_feature_relationships(
        shap_csv_path=str(level3 / "shap_values_long.csv"),
        output_dir=str(level3 / "feature_relationships"),
    )

    print(f"V2 interpretation data saved to {output_dir}")

In [24]:
# Загрузка данных
dataset_name = 'bankmarketing'
fold = 1
data = load_fold_data(dataset_name, fold)
config = get_dataset_config(dataset_name)
continuous_names = config["continuous"]
categorical_names = config["categorical"]

# Декодеры категорий
category_decoders = {}
for col in categorical_names:
    vocab = data["vocabulary"][col]
    decoder = {idx: val for val, idx in vocab.items()}
    category_decoders[col] = decoder

# Тестовые тензоры
X_cat_test = torch.tensor(data["X_test_cat"], dtype=torch.long)
X_cont_test = torch.tensor(data["X_test_cont"], dtype=torch.float32)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Список моделей для интерпретации с корректными параметрами
models_list = [
    # Первая версия (InterpretableTabTransformer)
    {
        "class": InterpretableTabTransformer,
        "path": Path("models_saved_2.0") / dataset_name / "KAN_additive" / f"fold{fold}_3.pt",
        "name": "KAN_additive",
        "kwargs": {
            "main_effect_type": "kan",
            "use_transformer": False,
            "embedding_dim": 64,
            "nhead": 8,
            "num_layers": 3,
            "kan_hidden_dim": 16,          # явно указываем 16
        },
        "v2": False,
        "output_dim": 1,                    # для бинарной классификации
    },
    {
        "class": InterpretableTabTransformer,
        "path": Path("models_saved_2.0") / dataset_name / "MLP_additive" / f"fold{fold}_3.pt",
        "name": "MLP_additive",
        "kwargs": {
            "main_effect_type": "mlp",
            "use_transformer": False,
            "embedding_dim": 64,
            "nhead": 8,
            "num_layers": 3,
        },
        "v2": False,
        "output_dim": 1,
    },
    {
        "class": InterpretableTabTransformer,
        "path": Path("models_saved_2.0") / dataset_name / "KAN_transformer" / f"fold{fold}_3.pt",
        "name": "KAN_transformer",
        "kwargs": {
            "main_effect_type": "kan",
            "use_transformer": True,
            "embedding_dim": 64,
            "nhead": 8,
            "num_layers": 3,
            "kan_hidden_dim": 16,
        },
        "v2": False,
        "output_dim": 1,
    },
    {
        "class": InterpretableTabTransformer,
        "path": Path("models_saved_2.0") / dataset_name / "MLP_transformer" / f"fold{fold}_3.pt",
        "name": "MLP_transformer",
        "kwargs": {
            "main_effect_type": "mlp",
            "use_transformer": True,
            "embedding_dim": 64,
            "nhead": 8,
            "num_layers": 3,
        },
        "v2": False,
        "output_dim": 1,
    },
    # V2 модель
    {
        "class": InterpretableTabTransformerV2,
        "path": Path("models_v2") / dataset_name / "interpretable_v2" / f"fold{fold}_v2.pt",
        "name": "interpretable_v2",
        "kwargs": {
            "embedding_dim": 64,
            "nhead": 8,
            "num_layers": 3,
            "kan_hidden_dim": 16,
        },
        "v2": True,
        "output_dim": 2,   # V2 обучали с output_dim=2
    }
]

for entry in models_list:
    model_path = entry["path"]
    if not model_path.exists():
        print(f"Model {entry['name']} not found at {model_path}, skipping.")
        continue

    print(f"\nProcessing {entry['name']}...")
    output_dim = entry["output_dim"]

    if entry["v2"]:
        model = entry["class"](
            output_dim=output_dim,
            vocabulary=data["vocabulary"],
            num_continuous_features=data["num_continuous"],
            **entry["kwargs"]
        )
        model.task = data["task"]
        model.load_state_dict(torch.load(model_path, map_location='cpu'), strict=False)
        model.to(device)
        model.eval()

        save_v2_interpretability(
            model=model,
            categorical_x=X_cat_test,
            continuous_x=X_cont_test,
            continuous_names=continuous_names,
            categorical_names=categorical_names,
            category_decoders=category_decoders,
            output_dir=f"results/interpretability/{dataset_name}_fold{fold}/{entry['name']}",
            device=device,
            n_points=300,
            shap_background_size=100,
            shap_explain_size=200,
        )
    else:
        model = entry["class"](
            output_dim=output_dim,
            vocabulary=data["vocabulary"],
            num_continuous_features=data["num_continuous"],
            **entry["kwargs"]
        )
        model.task = data["task"]
        model.load_state_dict(torch.load(model_path, map_location='cpu'), strict=False)
        model.to(device)
        model.eval()

        save_all_interpretability(
            model=model,
            categorical_x=X_cat_test,
            continuous_x=X_cont_test,
            continuous_names=continuous_names,
            categorical_names=categorical_names,
            category_decoders=category_decoders,
            output_dir=f"results/interpretability/{dataset_name}_fold{fold}/{entry['name']}",
            device=device,
            n_curve_points=300,
            shap_background_size=100,
            shap_explain_size=200,
        )

print("\nAll interpretation data saved.")


Processing KAN_additive...


D:\Data\Temp\ipykernel_14460\4163254801.py:141: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(model_path, map_location='cpu'), strict=False)

Saved KAN function: age
Saved KAN function: balance
Saved KAN function: duration
Saved KAN function: campaign
Saved KAN function: pdays
Saved KAN function: previous


PermutationExplainer explainer: 201it [00:28,  5.64it/s]                                                               
D:\Data\Temp\ipykernel_14460\4163254801.py:141: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues


Interpretation data saved to: results\interpretability\bankmarketing_fold1\KAN_additive

Processing MLP_additive...

Interpretation data saved to: results\interpretability\bankmarketing_fold1\MLP_additive

Processing KAN_transformer...
Saved KAN function: age
Saved KAN function: balance
Saved KAN function: duration
Saved KAN function: campaign
Saved KAN function: pdays
Saved KAN function: previous


...


Interpretation data saved to: results\interpretability\bankmarketing_fold1\KAN_transformer

Processing MLP_transformer...


PermutationExplainer explainer: 201it [00:46,  3.38it/s]                                                               
C:\Users\user\anaconda3\envs\rtx_env\lib\site-packages\torch\nn\modules\transformer.py:379: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(
D:\Data\Temp\ipykernel_14460\4163254801.py:116: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allow


Interpretation data saved to: results\interpretability\bankmarketing_fold1\MLP_transformer

Processing interpretable_v2...


PermutationExplainer explainer: 201it [01:02,  2.70it/s]                                                               

V2 interpretation data saved to results\interpretability\bankmarketing_fold1\interpretable_v2

All interpretation data saved.


In [25]:
import pandas as pd
import numpy as np
from pathlib import Path

# Базовые пути
base_dir = Path("results/interpretability/bankmarketing_fold1")
models = ["KAN_additive", "MLP_additive", "KAN_transformer", "MLP_transformer", "interpretable_v2"]

# Словарь для хранения данных
results = {}

for model in models:
    results[model] = {}
    
    # ----- Уровень 3: SHAP глобальная важность -----
    shap_path = base_dir / model / "level_3_shap" / "shap_global_importance.csv"
    if shap_path.exists():
        df_shap = pd.read_csv(shap_path)
        results[model]["shap"] = df_shap
    else:
        results[model]["shap"] = None
    
    # ----- Уровень 2: main effects (агрегированные) -----
    main_path = base_dir / model / "level_2_main_effects" / "main_effects.csv"
    if main_path.exists():
        df_main = pd.read_csv(main_path)
        # Агрегируем по признакам: средний абсолютный эффект
        agg_main = df_main.groupby(["feature", "feature_type"])["effect"].agg(["mean", "std"]).reset_index()
        results[model]["main_agg"] = agg_main
    else:
        results[model]["main_agg"] = None
    
    # ----- Уровень 2: категориальные эффекты -----
    cat_dir = base_dir / model / "level_2_main_effects" / "categorical"
    if cat_dir.exists():
        cat_files = list(cat_dir.glob("*_categorical_effects.csv"))
        cat_dict = {}
        for f in cat_files:
            feature_name = f.stem.replace("_categorical_effects", "")
            cat_dict[feature_name] = pd.read_csv(f)
        results[model]["cat_effects"] = cat_dict
    else:
        results[model]["cat_effects"] = {}

In [26]:
# ======================================================
# 1. Глобальная важность SHAP для каждой модели
# ======================================================
print("="*80)
print("GLOBAL SHAP IMPORTANCE (top-7 features)")
print("="*80)

for model in models:
    df = results[model]["shap"]
    if df is not None:
        print(f"\n**{model}**")
        display(df.head(7).style.format({"mean_abs_shap": "{:.4f}"}))
    else:
        print(f"\n**{model}** – SHAP data not found.")

GLOBAL SHAP IMPORTANCE (top-7 features)

**KAN_additive**


,feature,mean_abs_shap
0,duration,1.1040
1,contact,0.5944
2,month,0.4398
3,housing,0.3347
4,campaign,0.1620
5,pdays,0.1526
6,poutcome,0.1495



**MLP_additive**


,feature,mean_abs_shap
0,duration,1.1059
1,contact,0.4824
2,month,0.4072
3,housing,0.2951
4,day,0.1873
5,poutcome,0.1699
6,campaign,0.1691



**KAN_transformer**


,feature,mean_abs_shap
0,duration,1.4313
1,month,0.8290
2,contact,0.6202
3,day,0.3487
4,campaign,0.2137
5,balance,0.2067
6,previous,0.1983



**MLP_transformer**


,feature,mean_abs_shap
0,duration,1.3439
1,month,0.7279
2,contact,0.6195
3,day,0.4216
4,campaign,0.2627
5,marital,0.2201
6,poutcome,0.1981



**interpretable_v2**


,feature,mean_abs_shap
0,duration,0.8500
1,month,0.4368
2,contact,0.2846
3,day,0.2588
4,housing,0.2081
5,pdays,0.1849
6,education,0.1137


In [29]:
# ======================================================
# 2. Категориальные эффекты (для признаков job и marital)
# ======================================================
print("\n" + "="*80)
print("CATEGORICAL EFFECTS (for selected features)")
print("="*80)

cat_features = ["job", "marital"]  # можно добавить другие
for model in models:
    cat_dict = results[model]["cat_effects"]
    if cat_dict:
        print(f"\n**{model}**")
        for feat in cat_features:
            if feat in cat_dict:
                df = cat_dict[feat]
                # Сортируем по эффекту
                df_sorted = df.sort_values("effect", ascending=False)
                print(f"\n{feat}:")
                display(df_sorted.style.format({"effect": "{:.4f}"}))
            else:
                print(f"\n{feat}: data not found")
    else:
        print(f"\n**{model}** – categorical effects not found.")


CATEGORICAL EFFECTS (for selected features)

**KAN_additive**

job:


,category_id,category,effect
3,3,housemaid,2.0551
6,6,self-employed,1.6964
11,11,unknown,1.0874
5,5,retired,0.4297
1,1,blue-collar,0.4185
9,9,technician,0.1632
0,0,admin.,0.0297
2,2,entrepreneur,-0.0613
7,7,services,-0.1762
8,8,student,-0.3787



marital:


,category_id,category,effect
0,0,divorced,1.8675
2,2,single,0.9173
1,1,married,-0.0595



**MLP_additive**

job:


...


**MLP_transformer**

job:


,category_id,category,effect
6,6,self-employed,2.0111
11,11,unknown,0.9992
4,4,management,0.8739
8,8,student,0.8304
10,10,unemployed,0.5012
7,7,services,0.3284
0,0,admin.,-0.3120
9,9,technician,-0.5140
1,1,blue-collar,-0.5370
3,3,housemaid,-0.7782



marital:


,category_id,category,effect
1,1,married,1.6670
2,2,single,-0.2009
0,0,divorced,-0.2278



**interpretable_v2** – categorical effects not found.


In [27]:
# ======================================================
# 3. Средние абсолютные main‑effects для непрерывных признаков
# ======================================================
print("\n" + "="*80)
print("AVERAGE ABSOLUTE MAIN EFFECTS (continuous features)")
print("="*80)

for model in models:
    df = results[model]["main_agg"]
    if df is not None:
        cont_df = df[df["feature_type"] == "continuous"]
        if not cont_df.empty:
            print(f"\n**{model}**")
            # Сортируем по среднему абсолютному эффекту
            cont_df["mean_abs"] = cont_df["mean"].abs()
            cont_df_sorted = cont_df.sort_values("mean_abs", ascending=False)
            display(cont_df_sorted[["feature", "mean", "std"]].style.format({"mean": "{:.4f}", "std": "{:.4f}"}))
    else:
        print(f"\n**{model}** – main effects data not found.")

print("\nAll text results displayed.")


AVERAGE ABSOLUTE MAIN EFFECTS (continuous features)

**KAN_additive**


D:\Data\Temp\ipykernel_14460\3139277715.py:15: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  cont_df["mean_abs"] = cont_df["mean"].abs()


,feature,mean,std
6,duration,2.2917,2.2886
0,age,1.9932,0.5147
13,pdays,1.2770,0.5668
2,campaign,-1.1453,0.6057
1,balance,-0.8544,0.3274
15,previous,0.7759,0.1711



**MLP_additive**


D:\Data\Temp\ipykernel_14460\3139277715.py:15: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  cont_df["mean_abs"] = cont_df["mean"].abs()


...

,feature,mean,std
6,duration,-0.1642,0.1859
1,balance,0.0973,0.3051
2,campaign,0.0913,0.3303
0,age,-0.0796,0.1714
15,previous,0.0258,0.2611
13,pdays,-0.0191,0.0672



**MLP_transformer**


D:\Data\Temp\ipykernel_14460\3139277715.py:15: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  cont_df["mean_abs"] = cont_df["mean"].abs()


,feature,mean,std
6,duration,-1.1230,0.8478
15,previous,-0.2840,0.0544
0,age,-0.1912,0.0943
13,pdays,-0.0837,0.6149
2,campaign,-0.0356,0.2547
1,balance,0.0013,0.4511



**interpretable_v2** – main effects data not found.

All text results displayed.


In [28]:
from pathlib import Path

base_dir = Path("results/interpretability/bankmarketing_fold1")
models = ["KAN_additive", "MLP_additive", "KAN_transformer", "MLP_transformer", "interpretable_v2"]

for model in models:
    print(f"\n{'='*60}")
    print(f"Model: {model}")
    print(f"{'='*60}")
    
    # Level 1 intrinsic
    level1_dir = base_dir / model / "level_1_intrinsic"
    if level1_dir.exists():
        print(f"\nLevel 1 files in {level1_dir}:")
        for f in sorted(level1_dir.glob("*")):
            print(f"  {f.name}")
    else:
        # Maybe level_1_tokens for V2
        level1_alt = base_dir / model / "level_1_tokens"
        if level1_alt.exists():
            print(f"\nLevel 1 tokens in {level1_alt}:")
            for f in sorted(level1_alt.glob("*")):
                print(f"  {f.name}")
        else:
            print("  Level 1 directory not found")
    
    # Level 2
    level2_dir = base_dir / model / "level_2_main_effects"
    if level2_dir.exists():
        print(f"\nLevel 2 files in {level2_dir}:")
        for f in sorted(level2_dir.glob("*")):
            if f.is_dir():
                print(f"  {f.name}/")
            else:
                print(f"  {f.name}")
    
    # Level 3 SHAP
    level3_dir = base_dir / model / "level_3_shap"
    if level3_dir.exists():
        print(f"\nLevel 3 SHAP files in {level3_dir}:")
        for f in sorted(level3_dir.glob("*")):
            if f.is_dir():
                print(f"  {f.name}/")
            else:
                print(f"  {f.name}")
    print()


Model: KAN_additive

Level 1 files in results\interpretability\bankmarketing_fold1\KAN_additive\level_1_intrinsic:
  .ipynb_checkpoints
  age_KAN_function.csv
  age_KAN_internal
  balance_KAN_function.csv
  balance_KAN_internal
  campaign_KAN_function.csv
  campaign_KAN_internal
  duration_KAN_function.csv
  duration_KAN_internal
  pdays_KAN_function.csv
  pdays_KAN_internal
  previous_KAN_function.csv
  previous_KAN_internal

Level 2 files in results\interpretability\bankmarketing_fold1\KAN_additive\level_2_main_effects:
  categorical/
  main_effects.csv
  main_effects_predictions.csv

Level 3 SHAP files in results\interpretability\bankmarketing_fold1\KAN_additive\level_3_shap:
  .ipynb_checkpoints/
  feature_relationships/
  shap_global_importance.csv
  shap_values.npz
  shap_values_long.csv


Model: MLP_additive

Level 1 files in results\interpretability\bankmarketing_fold1\MLP_additive\level_1_intrinsic:
  .ipynb_checkpoints
  age_MLP_function.csv
  balance_MLP_function.csv
  camp